# ER-CyRIS — Paket B: Frozen-Gate Comparative Evaluation

Notebook ini membandingkan tiga sistem pada alert yang sama:

1. `detector_only`;
2. `detector_plus_xai` tanpa gate;
3. `full_er_cyris` dengan keputusan `accept`, `warn`, atau `abstain`.

**Kontrak anti-leakage:** threshold detektor berasal dari Paket A. Threshold gate dipilih **development-only**, kemudian dibekukan sebelum fixed internal evaluation set dibuka. Hasil fixed evaluation tidak pernah dipakai untuk menyesuaikan threshold.


## Urutan eksekusi wajib

1. Jalankan audit Paket A dan ekstraksi instance development.
2. Tinjau `gate_threshold_registry.csv`, lalu bekukan registry.
3. Pastikan `FROZEN_GATE_REGISTRY.json` ada dan cocok dengan checksum.
4. Baru ubah `RUN_FIXED_EVALUATION = True` dan jalankan evaluasi sekali.

Jika runtime terputus, checkpoint per konfigurasi memungkinkan proses dilanjutkan tanpa mengulang SHAP yang sudah selesai.


In [26]:
%pip install -q xgboost shap imbalanced-learn pyarrow

import gc
import hashlib
import inspect
import itertools
import json
import math
import os
import re
import resource
import time
import warnings
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import friedmanchisquare, spearmanr, wilcoxon
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
import shap

warnings.filterwarnings("ignore")
print("Imports ready")


Imports ready


In [27]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False

BASE = Path('/content/drive/MyDrive/ercyris_siklus2') if IN_COLAB else Path('.')
PACKAGE_A_DIR = (Path('/content/drive/MyDrive/ercyris_framework_validation/algorithmic_diversity_v1')
                 if IN_COLAB else Path('algorithmic_diversity_v1'))
OUT_DIR = (Path('/content/drive/MyDrive/ercyris_framework_validation/frozen_gate_v1')
           if IN_COLAB else Path('frozen_gate_v1'))
DEV_CHECKPOINTS = OUT_DIR / 'checkpoints_development'
EVAL_CHECKPOINTS = OUT_DIR / 'checkpoints_fixed_evaluation'
for folder in [OUT_DIR, DEV_CHECKPOINTS, EVAL_CHECKPOINTS]:
    folder.mkdir(parents=True, exist_ok=True)

PATHS = {
    'BGL': BASE / 'BGL.log', 'HDFS': BASE / 'HDFS.log',
    'HDFS_LABEL': BASE / 'anomaly_label.csv', 'UNSW-NB15': BASE / 'UNSW-NB15_1.csv',
    'CICIDS2018': BASE / '02-14-2018.csv',
}
PRIMARY_DATASETS = ['BGL', 'HDFS', 'UNSW-NB15']
INCLUDE_CICIDS_CONTRAST = False
SEEDS = [42, 52, 62, 72, 82]
REPRESENTATIONS = ['raw_telemetry_space', 'transformed_feature_space']
CONDITIONS = {'clean': 0.0, 'noise_sigma_0p05': 0.05}
MAX_SAMPLES_PER_DATASET, BGL_PARSE_LIMIT = 30_000, 250_000
MAX_ALERTS_PER_RUN, SHAP_BACKGROUND_N, TOP_K = 80, 40, 10
PROBE_SIGMA, MIN_ACCEPT_COVERAGE = 0.05, 0.50
RUN_DEVELOPMENT_GATE = True
# Tetap False sampai gate registry ditinjau dan dibekukan.
RUN_FIXED_EVALUATION = True

print('Package A:', PACKAGE_A_DIR)
print('Output:', OUT_DIR)
print('Primary datasets:', PRIMARY_DATASETS)
print('CICIDS secondary contrast:', INCLUDE_CICIDS_CONTRAST)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Package A: /content/drive/MyDrive/ercyris_framework_validation/algorithmic_diversity_v1
Output: /content/drive/MyDrive/ercyris_framework_validation/frozen_gate_v1
Primary datasets: ['BGL', 'HDFS', 'UNSW-NB15']
CICIDS secondary contrast: False


## Audit Paket A dan threshold detektor yang sudah dibekukan

In [28]:
PACKAGE_A_THRESHOLD = PACKAGE_A_DIR / 'threshold_registry.csv'
PACKAGE_A_FREEZE = PACKAGE_A_DIR / 'FROZEN_THRESHOLD_REGISTRY.json'
PACKAGE_A_HOLDOUT_LOCK = PACKAGE_A_DIR / 'HOLDOUT_EVALUATION_LOCK.json'

for required in [PACKAGE_A_THRESHOLD, PACKAGE_A_FREEZE, PACKAGE_A_HOLDOUT_LOCK]:
    if not required.exists():
        raise FileNotFoundError(f'Missing Package A artifact: {required}')

def freeze_registry_digest(records):
    canonical = sorted(json.dumps(record, sort_keys=True, separators=(',', ':')) for record in records)
    return hashlib.sha256('\n'.join(canonical).encode('utf-8')).hexdigest()

detector_registry = pd.read_csv(PACKAGE_A_THRESHOLD)
detector_records = detector_registry.sort_values('fit_id').to_dict('records')
detector_digest = freeze_registry_digest(detector_records)
package_a_freeze = json.loads(PACKAGE_A_FREEZE.read_text(encoding='utf-8'))
if detector_digest != package_a_freeze['sha256']:
    raise RuntimeError('Package A threshold registry checksum mismatch')
if set(detector_registry['model']) != set(['LogisticRegression', 'DecisionTree', 'RandomForest', 'XGBoost', 'SVC_RBF', 'MLP']):
    raise RuntimeError('Package A model set is incomplete')
print('Package A detector registry verified:', detector_digest)


Package A detector registry verified: b70a7a4dd26b42c9a5376d45c781cdf389e76827fbaecc077520fe7437b2bb30


In [29]:
"""Core utilities for the ER-CyRIS algorithmic-diversity benchmark."""

from __future__ import annotations

import hashlib
import json

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split


def _as_binary_vector(y) -> np.ndarray:
    values = np.asarray(y, dtype=int).reshape(-1)
    if len(values) < 8:
        raise ValueError("At least eight observations are required")
    if set(np.unique(values)) != {0, 1}:
        raise ValueError("Binary labels must contain both 0 and 1")
    return values


def make_four_way_split(y, seed: int = 42) -> dict[str, np.ndarray]:
    """Return disjoint 55/15/15/15 stratified index partitions."""
    labels = _as_binary_vector(y)
    indices = np.arange(len(labels))
    train_idx, remainder_idx = train_test_split(
        indices,
        train_size=0.55,
        random_state=seed,
        stratify=labels,
    )
    middle_idx, holdout_idx = train_test_split(
        remainder_idx,
        test_size=1.0 / 3.0,
        random_state=seed + 1,
        stratify=labels[remainder_idx],
    )
    calibration_idx, development_idx = train_test_split(
        middle_idx,
        test_size=0.5,
        random_state=seed + 2,
        stratify=labels[middle_idx],
    )
    return {
        "train": np.sort(train_idx),
        "calibration": np.sort(calibration_idx),
        "development": np.sort(development_idx),
        "holdout": np.sort(holdout_idx),
    }


def add_gaussian_noise(X, sigma: float, seed: int) -> np.ndarray:
    values = np.asarray(X, dtype=float)
    if sigma < 0:
        raise ValueError("sigma must be non-negative")
    if sigma == 0:
        return values.copy()
    noise = np.random.default_rng(seed).normal(0.0, sigma, size=values.shape)
    return np.clip(values + noise, 0.0, 1.0)


def expected_calibration_error(y_true, probabilities, n_bins: int = 10) -> float:
    labels = np.asarray(y_true, dtype=int).reshape(-1)
    probs = np.clip(np.asarray(probabilities, dtype=float).reshape(-1), 0.0, 1.0)
    if len(labels) != len(probs) or len(labels) == 0:
        raise ValueError("Labels and probabilities must have the same non-zero length")
    if n_bins < 1:
        raise ValueError("n_bins must be positive")

    edges = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.minimum(np.digitize(probs, edges[1:-1], right=False), n_bins - 1)
    ece = 0.0
    for bin_id in range(n_bins):
        mask = bin_ids == bin_id
        if not np.any(mask):
            continue
        accuracy = float(labels[mask].mean())
        confidence = float(probs[mask].mean())
        ece += float(mask.mean()) * abs(accuracy - confidence)
    return float(ece)


def select_f1_threshold(y_true, probabilities) -> tuple[float, float]:
    labels = np.asarray(y_true, dtype=int).reshape(-1)
    probs = np.clip(np.asarray(probabilities, dtype=float).reshape(-1), 0.0, 1.0)
    if len(labels) != len(probs) or len(labels) == 0:
        raise ValueError("Labels and probabilities must have the same non-zero length")

    candidates = np.unique(probs)
    scored = [
        (float(f1_score(labels, probs >= threshold, zero_division=0)), float(threshold))
        for threshold in candidates
    ]
    best_score = max(score for score, _ in scored)
    tied = [threshold for score, threshold in scored if score == best_score]
    best_threshold = min(tied, key=lambda value: (abs(value - 0.5), value))
    return float(best_threshold), float(best_score)


class PlattCalibrator:
    """One-dimensional sigmoid calibration fitted only on the calibration split."""

    def __init__(self, seed: int = 42):
        self.seed = int(seed)
        self.model = LogisticRegression(random_state=self.seed, solver="lbfgs")
        self._fitted = False

    def fit(self, scores, y):
        score_array = np.asarray(scores, dtype=float).reshape(-1, 1)
        labels = np.asarray(y, dtype=int).reshape(-1)
        if len(score_array) != len(labels) or set(np.unique(labels)) != {0, 1}:
            raise ValueError("Calibration data must contain aligned binary classes")
        self.model.fit(score_array, labels)
        self._fitted = True
        return self

    def predict_proba(self, scores) -> np.ndarray:
        if not self._fitted:
            raise RuntimeError("Call fit before predict_proba")
        score_array = np.asarray(scores, dtype=float).reshape(-1, 1)
        return self.model.predict_proba(score_array)[:, 1]


def stratified_cap_indices(y, max_samples: int, seed: int) -> np.ndarray:
    labels = _as_binary_vector(y)
    if max_samples < 2:
        raise ValueError("max_samples must be at least two")
    if len(labels) <= max_samples:
        return np.arange(len(labels), dtype=int)
    chosen, _ = train_test_split(
        np.arange(len(labels)),
        train_size=int(max_samples),
        random_state=seed,
        stratify=labels,
    )
    return np.sort(chosen)


def expected_run_count(
    n_datasets: int,
    n_models: int,
    n_representations: int,
    n_seeds: int,
    n_conditions: int,
) -> dict[str, int]:
    model_fits = int(n_datasets * n_models * n_representations * n_seeds)
    return {
        "model_fits": model_fits,
        "evaluation_rows": int(model_fits * n_conditions),
    }


def build_run_id(
    dataset: str,
    representation: str,
    model: str,
    seed: int,
    condition: str,
    split: str,
) -> str:
    return (
        f"{dataset}__{representation}__{model}__seed-{int(seed)}"
        f"__{condition}__{split}"
    )


def freeze_registry_digest(records: list[dict]) -> str:
    canonical_records = sorted(
        (json.dumps(record, sort_keys=True, separators=(",", ":")) for record in records)
    )
    payload = "\n".join(canonical_records).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()


In [30]:
"""Core, testable utilities for ER-CyRIS Package B."""

from __future__ import annotations

import hashlib
import json

import numpy as np
import pandas as pd


def detection_reliability(probability, threshold: float) -> np.ndarray:
    """Normalize positive-class margin to [0, 1]; non-alerts receive zero."""
    probability = np.clip(np.asarray(probability, dtype=float), 0.0, 1.0)
    threshold = float(threshold)
    if not 0.0 <= threshold < 1.0:
        raise ValueError("threshold must be in [0, 1)")
    return np.clip((probability - threshold) / max(1.0 - threshold, 1e-12), 0.0, 1.0)


def explanation_reliability(topk_jaccard, magnitude_stability, sign_consistency) -> np.ndarray:
    """Equal-weighted explanation reliability with bounded components."""
    components = np.column_stack(
        [topk_jaccard, magnitude_stability, sign_consistency]
    ).astype(float)
    return np.clip(components, 0.0, 1.0).mean(axis=1)


def instance_explanation_stability(clean_values, probe_values, top_k: int = 10) -> pd.DataFrame:
    """Compute Top-K, magnitude, sign, and composite stability per alert."""
    clean_values = np.asarray(clean_values, dtype=float)
    probe_values = np.asarray(probe_values, dtype=float)
    if clean_values.shape != probe_values.shape or clean_values.ndim != 2:
        raise ValueError("clean_values and probe_values must be same-shape 2-D arrays")
    k = min(max(int(top_k), 1), clean_values.shape[1])
    clean_abs = np.abs(clean_values)
    probe_abs = np.abs(probe_values)
    clean_top = np.argsort(-clean_abs, axis=1)[:, :k]
    probe_top = np.argsort(-probe_abs, axis=1)[:, :k]

    jaccard = []
    sign = []
    for i, (left, right) in enumerate(zip(clean_top, probe_top)):
        left_set, right_set = set(left.tolist()), set(right.tolist())
        union = left_set | right_set
        jaccard.append(len(left_set & right_set) / max(len(union), 1))
        sign_indices = sorted(union)
        sign.append(
            np.mean(
                np.sign(clean_values[i, sign_indices])
                == np.sign(probe_values[i, sign_indices])
            )
        )

    denominator = clean_abs.mean(axis=1) + 1e-12
    magnitude = 1.0 - np.abs(clean_abs - probe_abs).mean(axis=1) / denominator
    frame = pd.DataFrame(
        {
            "topk_jaccard": np.clip(jaccard, 0.0, 1.0),
            "magnitude_stability": np.clip(magnitude, 0.0, 1.0),
            "sign_consistency": np.clip(sign, 0.0, 1.0),
        }
    )
    frame["r_x"] = explanation_reliability(
        frame["topk_jaccard"],
        frame["magnitude_stability"],
        frame["sign_consistency"],
    )
    return frame


def assign_gate_action(r_d, r_x, tau_d: float, tau_x: float) -> np.ndarray:
    """Map two reliability domains to accept, warn, or abstain."""
    r_d = np.asarray(r_d, dtype=float)
    r_x = np.asarray(r_x, dtype=float)
    if r_d.shape != r_x.shape:
        raise ValueError("r_d and r_x must have the same shape")
    detection_ok = r_d >= float(tau_d)
    explanation_ok = r_x >= float(tau_x)
    return np.where(
        detection_ok & explanation_ok,
        "accept",
        np.where(detection_ok | explanation_ok, "warn", "abstain"),
    )


def fit_gate_thresholds(frame: pd.DataFrame, min_accept_coverage: float = 0.5) -> dict:
    """Fit gate thresholds only on development alerts using a fixed grid rule."""
    required = {"split", "r_d", "r_x", "y_true"}
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"missing columns: {sorted(missing)}")
    if frame.empty or set(frame["split"].astype(str)) != {"development"}:
        raise ValueError("gate threshold fitting requires development-only rows")
    if not 0.0 < min_accept_coverage <= 1.0:
        raise ValueError("min_accept_coverage must be in (0, 1]")

    quantiles = np.linspace(0.0, 1.0, 11)
    candidates_d = np.unique(np.quantile(frame["r_d"], quantiles))
    candidates_x = np.unique(np.quantile(frame["r_x"], quantiles))
    evaluated = []
    for tau_d in candidates_d:
        for tau_x in candidates_x:
            action = assign_gate_action(frame["r_d"], frame["r_x"], tau_d, tau_x)
            accepted = action == "accept"
            coverage = float(accepted.mean())
            if coverage + 1e-12 < min_accept_coverage or accepted.sum() == 0:
                continue
            unsafe_rate = float((frame.loc[accepted, "y_true"].to_numpy() == 0).mean())
            evaluated.append(
                {
                    "tau_d": float(tau_d),
                    "tau_x": float(tau_x),
                    "accept_coverage_development": coverage,
                    "unsafe_acceptance_development": unsafe_rate,
                    "n_development_alerts": int(len(frame)),
                }
            )
    if not evaluated:
        raise RuntimeError("no gate threshold pair satisfies minimum accept coverage")
    selected = min(
        evaluated,
        key=lambda row: (
            row["unsafe_acceptance_development"],
            -row["accept_coverage_development"],
            row["tau_d"] + row["tau_x"],
        ),
    )
    selected["selection_split"] = "development"
    selected["min_accept_coverage"] = float(min_accept_coverage)
    selected["grid_rule"] = "deciles; minimize unsafe acceptance; maximize coverage"
    return selected


def build_system_rows(alerts: pd.DataFrame, tau_d: float, tau_x: float) -> pd.DataFrame:
    """Create comparable per-alert rows for the three Package B systems."""
    required = {"alert_id", "y_true", "r_d", "r_x"}
    missing = required - set(alerts.columns)
    if missing:
        raise ValueError(f"missing columns: {sorted(missing)}")
    outputs = []
    for system in ["detector_only", "detector_plus_xai", "full_er_cyris"]:
        copy = alerts.copy()
        copy["system"] = system
        if system == "full_er_cyris":
            copy["action"] = assign_gate_action(copy["r_d"], copy["r_x"], tau_d, tau_x)
        else:
            copy["action"] = "accept"
        outputs.append(copy)
    return pd.concat(outputs, ignore_index=True)


def select_alert_indices(predictions, max_alerts: int, seed: int) -> np.ndarray:
    """Deterministically sample predicted-positive indices without replacement."""
    predictions = np.asarray(predictions, dtype=int)
    candidates = np.flatnonzero(predictions == 1)
    if len(candidates) <= int(max_alerts):
        return candidates
    rng = np.random.default_rng(int(seed))
    return np.sort(rng.choice(candidates, size=int(max_alerts), replace=False))


def freeze_registry_digest(records: list[dict]) -> str:
    """Create an order-invariant SHA-256 digest for a threshold registry."""
    canonical = sorted(
        json.dumps(record, sort_keys=True, separators=(",", ":")) for record in records
    )
    return hashlib.sha256("\n".join(canonical).encode("utf-8")).hexdigest()



In [31]:
class BaselineRepresentation:
    '''Median imputation plus min-max scaling fitted on train only.'''

    name = 'raw_telemetry_space'

    def __init__(self):
        self.imputer = SimpleImputer(strategy='median')
        self.scaler = MinMaxScaler()

    def fit(self, X):
        Xi = self.imputer.fit_transform(np.asarray(X, dtype=float))
        self.scaler.fit(Xi)
        return self

    def transform(self, X):
        Xi = self.imputer.transform(np.asarray(X, dtype=float))
        return np.clip(self.scaler.transform(Xi), 0.0, 1.0)

    def feature_names(self, names):
        return list(names)


class EvidenceAwareRepresentation(BaselineRepresentation):
    '''Variance weighting plus deviation features, rescaled on train only.'''

    name = 'transformed_feature_space'

    def __init__(self):
        super().__init__()
        self.idf = self.mu = self.sd = None
        self.final_scaler = MinMaxScaler()

    def _augment(self, base):
        weighted = base * (1.0 + 0.3 * self.idf)
        deviation = np.clip(np.abs(base - self.mu) / self.sd, 0.0, 5.0) / 5.0
        deviation = deviation[:, :min(5, deviation.shape[1])]
        return np.hstack([weighted, deviation])

    def fit(self, X):
        super().fit(X)
        base = super().transform(X)
        variance = np.var(base, axis=0) + 1e-9
        self.idf = np.log(1.0 / variance + 1.0)
        self.idf /= self.idf.max() + 1e-9
        self.mu = base.mean(axis=0)
        self.sd = base.std(axis=0) + 1e-9
        self.final_scaler.fit(self._augment(base))
        return self

    def transform(self, X):
        base = super().transform(X)
        return np.clip(self.final_scaler.transform(self._augment(base)), 0.0, 1.0)

    def feature_names(self, names):
        base = list(names)
        return base + [f'deviation_{name}' for name in base[:min(5, len(base))]]


REPRESENTATION_FACTORY = {
    'raw_telemetry_space': BaselineRepresentation,
    'transformed_feature_space': EvidenceAwareRepresentation,
}
print('Representations ready:', list(REPRESENTATION_FACTORY))


Representations ready: ['raw_telemetry_space', 'transformed_feature_space']


In [32]:
def stable_bucket(value, modulus=1000):
    digest = hashlib.blake2b(str(value).encode('utf-8'), digest_size=8).digest()
    return int.from_bytes(digest, 'big') % modulus


def load_bgl(path):
    severity = {
        'INFO': 1, 'DEBUG': 0, 'WARN': 2, 'WARNING': 2, 'ERROR': 3,
        'SEVERE': 4, 'FATAL': 4, 'APPFPO': 3, 'KERN': 2,
        'KERNSVC': 3, 'APPFPW': 2,
    }
    attack_words = [
        'fatal', 'error', 'fail', 'exception', 'abort', 'illegal',
        'segfault', 'panic', 'crash', 'out of memory', 'overflow',
    ]
    rows = []
    with open(path, 'r', errors='replace') as stream:
        for line_number, line in enumerate(stream):
            if BGL_PARSE_LIMIT and line_number >= BGL_PARSE_LIMIT:
                break
            parts = line.split(None, 8)
            if len(parts) < 9:
                continue
            label = 0 if parts[0] == '-' else 1
            sev = severity.get(parts[6].upper(), 1)
            message = parts[8].lower()
            try:
                hour = int(parts[3].split(':')[0]) if ':' in parts[3] else 12
            except Exception:
                hour = 12
            rows.append({
                'severity': sev,
                'is_error': int(sev >= 3),
                'is_warning': int(sev == 2),
                'hour': hour,
                'off_hour': int(hour < 6),
                'peak_hour': int(8 <= hour <= 18),
                'attack_keyword': int(any(word in message for word in attack_words)),
                'has_long_number': int(bool(re.search(r'\b\d{4,}\b', message))),
                'message_length': min(len(message), 500),
                'component_length': len(parts[5]),
                'node_hash': stable_bucket(parts[4]),
                'label': label,
            })
    frame = pd.DataFrame(rows)
    y = frame.pop('label').to_numpy(dtype=int)
    return frame.to_numpy(dtype=float), y, list(frame.columns)


def load_hdfs(log_path, label_path):
    line_pattern = re.compile(r'^(\d{6})\s+(\d{6})\s+\d+\s+(\w+)\s+([^:]+):\s+(.*)')
    block_pattern = re.compile(r'(blk_-?\d+)')
    severity = {'DEBUG': 0, 'INFO': 1, 'WARN': 2, 'WARNING': 2, 'ERROR': 3, 'FATAL': 4}
    blocks = defaultdict(lambda: {'n': 0, 'error': 0, 'warn': 0, 'components': set(), 'max': 0, 'sum': 0})
    with open(log_path, 'r', errors='replace') as stream:
        for line in stream:
            match = line_pattern.match(line)
            if not match:
                continue
            _, _, level, component, message = match.groups()
            sev = severity.get(level.upper(), 1)
            for block in block_pattern.findall(message):
                row = blocks[block]
                row['n'] += 1
                row['components'].add(component.strip())
                row['max'] = max(row['max'], sev)
                row['sum'] += sev
                row['error'] += int(sev >= 3)
                row['warn'] += int(sev == 2)
    labels = pd.read_csv(label_path)
    label_map = dict(zip(
        labels['BlockId'].astype(str),
        (labels['Label'].str.upper() == 'ANOMALY').astype(int),
    ))
    rows = []
    for block, row in blocks.items():
        if block not in label_map:
            continue
        count = max(row['n'], 1)
        rows.append({
            'n_events': row['n'],
            'n_error': row['error'],
            'n_warn': row['warn'],
            'n_components': len(row['components']),
            'max_severity': row['max'],
            'mean_severity': row['sum'] / count,
            'error_ratio': row['error'] / count,
            'warn_ratio': row['warn'] / count,
            'label': label_map[block],
        })
    frame = pd.DataFrame(rows)
    y = frame.pop('label').to_numpy(dtype=int)
    return frame.to_numpy(dtype=float), y, list(frame.columns)


def load_unsw(path):
    frame = pd.read_csv(path, low_memory=False)
    frame.columns = [column.lower().strip() for column in frame.columns]
    label_column = next((c for c in ['label', 'is_attack'] if c in frame.columns), frame.columns[-1])
    y = (pd.to_numeric(frame[label_column], errors='coerce').fillna(0) != 0).astype(int).to_numpy()
    frame.drop(columns=[label_column, 'srcip', 'dstip', 'attack_cat'], errors='ignore', inplace=True)
    for column in ['proto', 'state', 'service']:
        if column in frame.columns:
            frame[column] = LabelEncoder().fit_transform(frame[column].fillna('unknown').astype(str))
    frame = frame.apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan)
    return frame.to_numpy(dtype=float), y, list(frame.columns)


def load_cicids(path):
    frame = pd.read_csv(path, low_memory=False)
    label_column = next(column for column in frame.columns if 'label' in column.lower())
    y = (frame[label_column].astype(str).str.strip().str.upper() != 'BENIGN').astype(int).to_numpy()
    frame.drop(columns=[label_column], inplace=True)
    frame.drop(columns=[
        column for column in frame.columns
        if any(key in column.lower() for key in ['timestamp', 'flow id', 'src ip', 'dst ip'])
    ], errors='ignore', inplace=True)
    frame.columns = [column.strip().lower().replace(' ', '_').replace('/', '_per_') for column in frame.columns]
    frame = frame.apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan)
    frame['__label__'] = y
    features = [column for column in frame.columns if column != '__label__']
    frame = frame.drop_duplicates(subset=features, keep='first').reset_index(drop=True)
    y = frame.pop('__label__').to_numpy(dtype=int)
    return frame.to_numpy(dtype=float), y, list(frame.columns)


def load_requested_datasets():
    requested = list(PRIMARY_DATASETS)
    if INCLUDE_CICIDS_CONTRAST:
        requested.append('CICIDS2018')
    loaded = {}
    for name in requested:
        if name == 'BGL':
            required = [PATHS['BGL']]
            loader = lambda: load_bgl(PATHS['BGL'])
        elif name == 'HDFS':
            required = [PATHS['HDFS'], PATHS['HDFS_LABEL']]
            loader = lambda: load_hdfs(PATHS['HDFS'], PATHS['HDFS_LABEL'])
        elif name == 'UNSW-NB15':
            required = [PATHS['UNSW-NB15']]
            loader = lambda: load_unsw(PATHS['UNSW-NB15'])
        else:
            required = [PATHS['CICIDS2018']]
            loader = lambda: load_cicids(PATHS['CICIDS2018'])
        missing = [str(path) for path in required if not path.exists()]
        if missing:
            raise FileNotFoundError(f'{name}: missing {missing}')
        X, y, names = loader()
        cap = stratified_cap_indices(y, MAX_SAMPLES_PER_DATASET, seed=42)
        loaded[name] = (X[cap], y[cap], names)
        print(f'{name}: {len(cap):,} rows, {X.shape[1]} features, anomaly={y[cap].mean():.3%}')
    return loaded


In [33]:
MODEL_NAMES = [
    'LogisticRegression',
    'DecisionTree',
    'RandomForest',
    'XGBoost',
    'SVC_RBF',
    'MLP',
]


def make_model(name, seed):
    registry = {
        'LogisticRegression': LogisticRegression(
            max_iter=1000, solver='liblinear', random_state=seed,
        ),
        'DecisionTree': DecisionTreeClassifier(
            min_samples_leaf=2, random_state=seed,
        ),
        'RandomForest': RandomForestClassifier(
            n_estimators=300, min_samples_leaf=2, n_jobs=-1, random_state=seed,
        ),
        'XGBoost': XGBClassifier(
            n_estimators=300, max_depth=6, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8, tree_method='hist',
            eval_metric='logloss', n_jobs=-1, random_state=seed, verbosity=0,
        ),
        'SVC_RBF': SVC(
            kernel='rbf', probability=False, cache_size=4000, random_state=seed,
        ),
        'MLP': MLPClassifier(
            hidden_layer_sizes=(128, 64), early_stopping=True,
            max_iter=150, batch_size='auto', random_state=seed,
        ),
    }
    return registry[name]


def fit_with_balancing(model, X, y, seed):
    weights = compute_sample_weight(class_weight='balanced', y=y)
    if 'sample_weight' in inspect.signature(model.fit).parameters:
        model.fit(X, y, sample_weight=weights)
        return model
    # Compatibility fallback for estimators without sample_weight.
    rng = np.random.default_rng(seed)
    class_indices = [np.flatnonzero(y == label) for label in (0, 1)]
    target = max(len(indices) for indices in class_indices)
    balanced = np.concatenate([
        rng.choice(indices, size=target, replace=len(indices) < target)
        for indices in class_indices
    ])
    rng.shuffle(balanced)
    model.fit(X[balanced], y[balanced])
    return model


def raw_model_score(model, X):
    if hasattr(model, 'decision_function'):
        return np.asarray(model.decision_function(X), dtype=float).reshape(-1)
    probability = np.asarray(model.predict_proba(X), dtype=float)[:, 1]
    probability = np.clip(probability, 1e-6, 1.0 - 1e-6)
    return np.log(probability / (1.0 - probability))


print('Models ready:', MODEL_NAMES)


Models ready: ['LogisticRegression', 'DecisionTree', 'RandomForest', 'XGBoost', 'SVC_RBF', 'MLP']


## Checkpoint, SHAP, dan ekstraksi reliabilitas per alert

In [34]:
DEV_INSTANCES = OUT_DIR / 'development_gate_instances.parquet'
GATE_REGISTRY = OUT_DIR / 'gate_threshold_registry.csv'
GATE_FREEZE = OUT_DIR / 'FROZEN_GATE_REGISTRY.json'
EVAL_INSTANCES = OUT_DIR / 'fixed_evaluation_gate_instances.parquet'
SYSTEM_ROWS = OUT_DIR / 'three_system_per_alert_results.parquet'
SUMMARY = OUT_DIR / 'three_system_summary.csv'
SENSITIVITY = OUT_DIR / 'gate_sensitivity.csv'
LOCK_PATH = OUT_DIR / 'GATE_EVALUATION_LOCK.json'
MANIFEST_PATH = OUT_DIR / 'package_b_manifest.json'

def atomic_csv(frame, path):
    temporary = path.with_suffix(path.suffix + '.tmp')
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)

def atomic_json(payload, path):
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(payload, indent=2), encoding='utf-8')
    os.replace(temporary, path)

def normalize_shap(values):
    if isinstance(values, list): values = values[-1]
    values = np.asarray(values)
    if values.ndim == 3: values = values[:, :, -1]
    if values.ndim != 2: raise ValueError(f'Expected 2-D SHAP values, got {values.shape}')
    return values.astype(float)

def explain_model(model_name, model, calibrator, background, samples):
    if model_name in {'DecisionTree', 'RandomForest', 'XGBoost'}:
        return normalize_shap(shap.TreeExplainer(model).shap_values(samples))
    if model_name == 'LogisticRegression':
        return normalize_shap(shap.LinearExplainer(model, background).shap_values(samples))
    def calibrated_predict(matrix):
        return calibrator.predict_proba(raw_model_score(model, np.asarray(matrix)))
    explainer = shap.Explainer(calibrated_predict, background, algorithm='permutation')
    result = explainer(samples, max_evals=max(2 * samples.shape[1] + 1, 101), silent=True)
    return normalize_shap(result.values)

def threshold_for(dataset, representation, model, seed):
    match = detector_registry.query(
        'dataset == @dataset and representation == @representation and model == @model and seed == @seed'
    )
    if len(match) != 1: raise RuntimeError('Detector threshold lookup must return exactly one row')
    return float(match.iloc[0]['threshold'])

def check_package_a_reproduction(run_id, split_name, y_true, probability, prediction):
    folder = 'predictions_development' if split_name == 'development' else 'predictions_holdout'
    reference_path = PACKAGE_A_DIR / folder / f'{run_id}.parquet'
    if not reference_path.exists():
        raise FileNotFoundError(f'Missing Package A prediction evidence: {reference_path}')
    reference = pd.read_parquet(reference_path).sort_values('instance_id')
    if len(reference) != len(y_true) or not np.array_equal(reference['y_true'].to_numpy(), y_true):
        raise RuntimeError(f'Package A label/row reproduction failed: {run_id}')
    probability_mae = float(np.mean(np.abs(reference['probability'].to_numpy() - probability)))
    legacy_prediction_agreement = float(np.mean(reference['prediction'].to_numpy() == prediction))
    if probability_mae > 0.01:
        raise RuntimeError(f'Package A probability reproduction failed: {run_id}; MAE={probability_mae:.6f}')
    if legacy_prediction_agreement < 0.995:
        print(f'DIAGNOSTIC: legacy prediction mismatch for {run_id}; '
              f'probability MAE={probability_mae:.6f}, agreement={legacy_prediction_agreement:.6f}. '
              'Proceeding with reproduced probability and frozen detector threshold.')
    return probability_mae, legacy_prediction_agreement


In [35]:
def extract_split_instances(datasets, split_name, checkpoint_dir):
    rows = []
    for dataset, (X, y, raw_names) in datasets.items():
        for seed in SEEDS:
            split = make_four_way_split(y, seed)
            for representation in REPRESENTATIONS:
                transformer = REPRESENTATION_FACTORY[representation]().fit(X[split['train']])
                X_train = transformer.transform(X[split['train']])
                X_cal = transformer.transform(X[split['calibration']])
                X_eval_base = transformer.transform(X[split[split_name]])
                y_train, y_cal, y_eval = y[split['train']], y[split['calibration']], y[split[split_name]]
                names = transformer.feature_names(raw_names)
                for model_name in MODEL_NAMES:
                    model = fit_with_balancing(make_model(model_name, seed), X_train, y_train, seed)
                    calibrator = PlattCalibrator(seed).fit(raw_model_score(model, X_cal), y_cal)
                    threshold = threshold_for(dataset, representation, model_name, seed)
                    rng = np.random.default_rng(seed + 9000)
                    bg_idx = rng.choice(len(X_train), min(SHAP_BACKGROUND_N, len(X_train)), replace=False)
                    background = X_train[bg_idx]
                    for condition, sigma in CONDITIONS.items():
                        run_id = build_run_id(dataset, representation, model_name, seed, condition, split_name)
                        checkpoint = checkpoint_dir / f'{run_id}.parquet'
                        if checkpoint.exists():
                            rows.append(pd.read_parquet(checkpoint)); continue
                        noise_seed = seed + (1000 if split_name == 'development' else 2000)
                        X_eval = add_gaussian_noise(X_eval_base, sigma, noise_seed)
                        probability = calibrator.predict_proba(raw_model_score(model, X_eval))
                        prediction = (probability >= threshold).astype(int)
                        reproduction_mae, legacy_prediction_agreement = check_package_a_reproduction(
                            run_id, split_name, y_eval, probability, prediction)
                        selected = select_alert_indices(prediction, MAX_ALERTS_PER_RUN, seed + stable_bucket(run_id, 100000))
                        if len(selected) == 0:
                            print('No alerts:', run_id); continue
                        samples = X_eval[selected]
                        probe_seed = seed + (6000 if split_name == 'development' else 7000) + stable_bucket(run_id, 100000)
                        probes = add_gaussian_noise(samples, PROBE_SIGMA, probe_seed)
                        started = time.perf_counter()
                        current_shap = explain_model(model_name, model, calibrator, background, samples)
                        probe_shap = explain_model(model_name, model, calibrator, background, probes)
                        shap_seconds = time.perf_counter() - started
                        stability = instance_explanation_stability(current_shap, probe_shap, TOP_K)
                        frame = stability.assign(
                            alert_id=[f'{run_id}__instance-{int(i)}' for i in selected],
                            instance_id=selected, y_true=y_eval[selected], probability=probability[selected],
                            detector_threshold=threshold,
                            r_d=detection_reliability(probability[selected], threshold),
                            dataset=dataset, representation=representation, model=model_name, seed=seed,
                            condition=condition, split=split_name, run_id=run_id,
                            shap_seconds=shap_seconds, feature_count=len(names),
                            reproduction_probability_mae=reproduction_mae,
                            legacy_prediction_agreement=legacy_prediction_agreement,
                        )
                        temporary = checkpoint.with_suffix('.tmp.parquet')
                        frame.to_parquet(temporary, index=False); os.replace(temporary, checkpoint)
                        rows.append(frame)
                        print('Saved', run_id, len(frame), 'alerts')
                    del model, calibrator
                    gc.collect()
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

DATASETS = load_requested_datasets()
if RUN_DEVELOPMENT_GATE:
    development_instances = extract_split_instances(DATASETS, 'development', DEV_CHECKPOINTS)
    development_instances.to_parquet(DEV_INSTANCES, index=False)
    print('Development alert instances:', len(development_instances))


BGL: 30,000 rows, 11 features, anomaly=12.023%
HDFS: 30,000 rows, 8 features, anomaly=2.927%
UNSW-NB15: 30,000 rows, 48 features, anomaly=3.173%
Development alert instances: 27787


## Pilih threshold gate dari development saja

In [36]:
if not DEV_INSTANCES.exists(): raise FileNotFoundError('Run development extraction first')
development_instances = pd.read_parquet(DEV_INSTANCES)
if set(development_instances['split']) != {'development'}:
    raise RuntimeError('Gate fitting input is not development-only')

gate_rows, sensitivity_rows = [], []
group_columns = ['dataset', 'representation', 'model']
for keys, group in development_instances.groupby(group_columns, sort=True):
    selected = fit_gate_thresholds(group, MIN_ACCEPT_COVERAGE)
    gate_rows.append(dict(zip(group_columns, keys), **selected))
    for tau_d in np.unique(np.quantile(group['r_d'], np.linspace(0, 1, 11))):
        for tau_x in np.unique(np.quantile(group['r_x'], np.linspace(0, 1, 11))):
            action = assign_gate_action(group['r_d'], group['r_x'], tau_d, tau_x)
            accepted = action == 'accept'
            sensitivity_rows.append(dict(zip(group_columns, keys), tau_d=tau_d, tau_x=tau_x,
                accept_coverage=accepted.mean(), unsafe_acceptance=(group.loc[accepted, 'y_true'] == 0).mean() if accepted.any() else np.nan,
                selection_split='development'))

gate_registry = pd.DataFrame(gate_rows)
if len(gate_registry) != len(PRIMARY_DATASETS) * len(REPRESENTATIONS) * len(MODEL_NAMES):
    raise RuntimeError('Incomplete gate registry; expected one row per dataset x representation x model')
atomic_csv(gate_registry, GATE_REGISTRY)
atomic_csv(pd.DataFrame(sensitivity_rows), SENSITIVITY)
display(gate_registry)


,dataset,representation,model,tau_d,tau_x,accept_coverage_development,unsafe_acceptance_development,n_development_alerts,selection_split,min_accept_coverage,grid_rule
0,BGL,raw_telemetry_space,DecisionTree,0.990101,0.600000,0.531250,0.152941,800,development,0.5,deciles; minimize unsafe acceptance; maximize ...
1,BGL,raw_telemetry_space,LogisticRegression,0.401118,0.780092,0.800000,0.145313,800,development,0.5,deciles; minimize unsafe acceptance; maximize ...
2,BGL,raw_telemetry_space,MLP,0.969016,0.385082,0.500000,0.060000,800,development,0.5,deciles; minimize unsafe acceptance; maximize ...
3,BGL,raw_telemetry_space,RandomForest,0.889176,0.566667,0.700000,0.053571,800,development,0.5,deciles; minimize unsafe acceptance; maximize ...
4,BGL,raw_telemetry_space,SVC_RBF,0.000000,0.734596,0.700000,0.060714,800,development,0.5,deciles; minimize unsafe acceptance; maximize ...
5,BGL,raw_telemetry_space,XGBoost,0.962538,0.692733,0.549858,0.067358,702,development,0.5,deciles; minimize unsafe acceptance; maximize ...
6,BGL,transformed_feature_space,DecisionTree,0.990101,0.600000,0.552500,0.140271,800,development,0.5,deciles; minimize unsafe acceptance; maximize ...
7,BGL,transformed_feature_space,LogisticRegression,0.454211,0.784465,0.800000,0.137500,800,development,0.5,deciles; minimize unsafe acceptance; maximize ...
8,BGL,transformed_feature_space,MLP,0.967906,0.291509,0.500000,0.072500,800,development,0.5,deciles; minimize unsafe acceptance; maximize ...
9,BGL,transformed_feature_space,RandomForest,0.973882,0.566667,0.600000,0.072917,800,development,0.5,deciles; minimize unsafe acceptance; maximize ...


## Bekukan registry gate sebelum fixed evaluation

In [12]:
gate_registry = pd.read_csv(GATE_REGISTRY)
if set(gate_registry['selection_split']) != {'development'}:
    raise RuntimeError('Refusing to freeze a non-development gate registry')
gate_records = gate_registry.sort_values(['dataset', 'representation', 'model']).to_dict('records')
gate_digest = freeze_registry_digest(gate_records)
freeze_payload = {
    'status': 'FROZEN', 'threshold_rows': len(gate_records), 'sha256': gate_digest,
    'selection_split': 'development', 'minimum_accept_coverage': MIN_ACCEPT_COVERAGE,
    'rule': 'minimize unsafe acceptance, then maximize coverage; fixed decile grid',
    'fixed_evaluation_used_for_tuning': False,
}
atomic_json(freeze_payload, GATE_FREEZE)
print('Frozen gate registry:', gate_digest)


Frozen gate registry: 8484cc9d96d9a9f71d397ae5c21d72621114c958bf1c19fd058d79fddae41374


## Fixed internal evaluation set

Ini bukan lagi disebut *untouched holdout* untuk Paket B karena hasil agregat Paket A sudah pernah dilihat. Evaluasi ini tetap sah sebagai **fixed internal evaluation set**, selama threshold gate yang dibekukan di atas tidak diubah berdasarkan hasilnya.


In [37]:
def verify_frozen_gate():
    if not GATE_FREEZE.exists(): raise RuntimeError('Freeze gate registry before evaluation')
    frozen = json.loads(GATE_FREEZE.read_text(encoding='utf-8'))
    current = pd.read_csv(GATE_REGISTRY).sort_values(['dataset', 'representation', 'model']).to_dict('records')
    if frozen['sha256'] != freeze_registry_digest(current):
        raise RuntimeError('Gate registry changed after freeze')
    if frozen['selection_split'] != 'development':
        raise RuntimeError('Invalid gate selection split')
    return frozen, pd.DataFrame(current)

def summarize_systems(system_rows):
    output = []
    group_cols = ['dataset', 'representation', 'model', 'condition', 'system']
    for keys, group in system_rows.groupby(group_cols, sort=True):
        accepted = group['action'] == 'accept'
        positives = group['y_true'] == 1
        output.append(dict(zip(group_cols, keys), n_alerts=len(group),
            accept_coverage=accepted.mean(), warn_rate=(group['action'] == 'warn').mean(),
            abstain_rate=(group['action'] == 'abstain').mean(),
            unsafe_acceptance=((group['y_true'] == 0) & accepted).sum() / max(accepted.sum(), 1),
            accepted_precision=group.loc[accepted, 'y_true'].mean() if accepted.any() else np.nan,
            retained_tp=(accepted & positives).sum() / max(positives.sum(), 1),
            mean_shap_seconds=group['shap_seconds'].mean()))
    summary = pd.DataFrame(output)
    baseline = summary.query("system == 'detector_only'")[group_cols[:-1] + ['unsafe_acceptance']].rename(
        columns={'unsafe_acceptance': 'baseline_unsafe_acceptance'})
    summary = summary.merge(baseline, on=group_cols[:-1], how='left')
    summary['false_accept_reduction'] = summary['baseline_unsafe_acceptance'] - summary['unsafe_acceptance']
    summary['conditional_decision_cost'] = (1-summary['accept_coverage']) + 2*summary['unsafe_acceptance']
    return summary

if RUN_FIXED_EVALUATION:
    frozen_gate, frozen_registry = verify_frozen_gate()
    fixed_instances = extract_split_instances(DATASETS, 'holdout', EVAL_CHECKPOINTS)
    fixed_instances['split'] = 'fixed_internal_evaluation'
    fixed_instances.to_parquet(EVAL_INSTANCES, index=False)
    systems = []
    for keys, group in fixed_instances.groupby(['dataset', 'representation', 'model'], sort=True):
        match = frozen_registry[(frozen_registry['dataset'] == keys[0]) &
                                (frozen_registry['representation'] == keys[1]) &
                                (frozen_registry['model'] == keys[2])]
        if len(match) != 1: raise RuntimeError('Frozen gate lookup failed')
        systems.append(build_system_rows(group, float(match.iloc[0].tau_d), float(match.iloc[0].tau_x)))
    system_rows = pd.concat(systems, ignore_index=True)
    system_rows.to_parquet(SYSTEM_ROWS, index=False)
    summary = summarize_systems(system_rows)
    atomic_csv(summary, SUMMARY)
    lock = {'status': 'EVALUATED_ONCE', 'gate_registry_sha256': frozen_gate['sha256'],
            'selection_split': 'development', 'evaluation_split': 'fixed_internal_evaluation',
            'fixed_evaluation_used_for_tuning': False, 'rows': len(system_rows)}
    atomic_json(lock, LOCK_PATH)
    display(summary)
else:
    print('Fixed evaluation NOT run. Review and freeze the development gate registry first.')


Saved BGL__raw_telemetry_space__LogisticRegression__seed-42__clean__holdout 80 alerts
Saved BGL__raw_telemetry_space__LogisticRegression__seed-42__noise_sigma_0p05__holdout 80 alerts
Saved BGL__raw_telemetry_space__DecisionTree__seed-42__clean__holdout 80 alerts
Saved BGL__raw_telemetry_space__DecisionTree__seed-42__noise_sigma_0p05__holdout 80 alerts
Saved BGL__raw_telemetry_space__RandomForest__seed-42__clean__holdout 80 alerts
Saved BGL__raw_telemetry_space__RandomForest__seed-42__noise_sigma_0p05__holdout 80 alerts
Saved BGL__raw_telemetry_space__XGBoost__seed-42__clean__holdout 80 alerts
Saved BGL__raw_telemetry_space__XGBoost__seed-42__noise_sigma_0p05__holdout 59 alerts
Saved BGL__raw_telemetry_space__SVC_RBF__seed-42__clean__holdout 80 alerts
Saved BGL__raw_telemetry_space__SVC_RBF__seed-42__noise_sigma_0p05__holdout 80 alerts
Saved BGL__raw_telemetry_space__MLP__seed-42__clean__holdout 80 alerts
Saved BGL__raw_telemetry_space__MLP__seed-42__noise_sigma_0p05__holdout 80 alerts


,dataset,representation,model,condition,system,n_alerts,accept_coverage,warn_rate,abstain_rate,unsafe_acceptance,accepted_precision,retained_tp,mean_shap_seconds,baseline_unsafe_acceptance,false_accept_reduction,conditional_decision_cost
0,BGL,raw_telemetry_space,DecisionTree,clean,detector_only,400,1.0000,0.0000,0.0,0.002500,0.997500,1.000000,0.002165,0.0025,0.000000,0.005000
1,BGL,raw_telemetry_space,DecisionTree,clean,detector_plus_xai,400,1.0000,0.0000,0.0,0.002500,0.997500,1.000000,0.002165,0.0025,0.000000,0.005000
2,BGL,raw_telemetry_space,DecisionTree,clean,full_er_cyris,400,0.6000,0.4000,0.0,0.004167,0.995833,0.598997,0.002165,0.0025,-0.001667,0.408333
3,BGL,raw_telemetry_space,DecisionTree,noise_sigma_0p05,detector_only,400,1.0000,0.0000,0.0,0.400000,0.600000,1.000000,0.002028,0.4000,0.000000,0.800000
4,BGL,raw_telemetry_space,DecisionTree,noise_sigma_0p05,detector_plus_xai,400,1.0000,0.0000,0.0,0.400000,0.600000,1.000000,0.002028,0.4000,0.000000,0.800000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
211,UNSW-NB15,transformed_feature_space,XGBoost,clean,detector_plus_xai,400,1.0000,0.0000,0.0,0.122500,0.877500,1.000000,0.164589,0.1225,0.000000,0.245000
212,UNSW-NB15,transformed_feature_space,XGBoost,clean,full_er_cyris,400,0.7625,0.2375,0.0,0.019672,0.980328,0.851852,0.164589,0.1225,0.102828,0.276844
213,UNSW-NB15,transformed_feature_space,XGBoost,noise_sigma_0p05,detector_only,400,1.0000,0.0000,0.0,0.152500,0.847500,1.000000,0.234403,0.1525,0.000000,0.305000
214,UNSW-NB15,transformed_feature_space,XGBoost,noise_sigma_0p05,detector_plus_xai,400,1.0000,0.0000,0.0,0.152500,0.847500,1.000000,0.234403,0.1525,0.000000,0.305000


## Bootstrap perbandingan berpasangan dan manifest

In [38]:
def paired_bootstrap_full_vs_detector(system_rows, n_boot=2000, seed=20260923):
    wide = system_rows.pivot_table(index='alert_id', columns='system', values=['y_true', 'action'], aggfunc='first')
    rng = np.random.default_rng(seed); estimates = []
    ids = np.arange(len(wide))
    for _ in range(n_boot):
        sample = wide.iloc[rng.choice(ids, len(ids), replace=True)]
        y = sample[('y_true', 'full_er_cyris')].to_numpy()
        full_accept = sample[('action', 'full_er_cyris')].to_numpy() == 'accept'
        base_unsafe = (y == 0).mean()
        full_unsafe = ((y == 0) & full_accept).sum() / max(full_accept.sum(), 1)
        estimates.append(base_unsafe - full_unsafe)
    return {'false_accept_reduction_mean': float(np.mean(estimates)),
            'ci_2p5': float(np.quantile(estimates, .025)), 'ci_97p5': float(np.quantile(estimates, .975))}

manifest = {
    'package': 'B: Frozen-Gate Comparative Evaluation',
    'systems': ['detector_only', 'detector_plus_xai', 'full_er_cyris'],
    'actions': ['accept', 'warn', 'abstain'], 'models': MODEL_NAMES,
    'datasets': PRIMARY_DATASETS, 'representations': REPRESENTATIONS, 'seeds': SEEDS,
    'gate_selection': 'development-only', 'fixed_evaluation_used_for_tuning': False,
    'cross_model_agreement_role': 'diagnostic_only_not_a_hard_gate',
    'sample_scope': 'maximum 80 predicted alerts per configuration',
    'package_a_detector_registry_sha256': detector_digest,
    'gate_registry_sha256': json.loads(GATE_FREEZE.read_text())['sha256'] if GATE_FREEZE.exists() else None,
}
atomic_json(manifest, MANIFEST_PATH)
if RUN_FIXED_EVALUATION and SYSTEM_ROWS.exists():
    print(paired_bootstrap_full_vs_detector(pd.read_parquet(SYSTEM_ROWS)))
print('Manifest:', MANIFEST_PATH)


{'false_accept_reduction_mean': 0.09408700452555854, 'ci_2p5': 0.08972495866264744, 'ci_97p5': 0.09846388633333805}
Manifest: /content/drive/MyDrive/ercyris_framework_validation/frozen_gate_v1/package_b_manifest.json


In [39]:
# ============================================================
# CORRECTED PACKAGE B POST-PROCESSING
# Tidak melatih ulang model dan tidak mengubah frozen gate
# ============================================================

import json
import os
from pathlib import Path

import numpy as np
import pandas as pd


SYSTEM_ROWS = OUT_DIR / "three_system_per_alert_results.parquet"

CORRECTED_SUMMARY = (
    OUT_DIR / "three_system_summary_corrected.csv"
)

CORRECTED_BOOTSTRAP = (
    OUT_DIR / "package_b_cluster_bootstrap_corrected.json"
)

ANALYSIS_LOCK = (
    OUT_DIR / "PACKAGE_B_CORRECTED_ANALYSIS_LOCK.json"
)


if not SYSTEM_ROWS.exists():
    raise FileNotFoundError(
        "three_system_per_alert_results.parquet belum tersedia"
    )


system_rows = pd.read_parquet(SYSTEM_ROWS)

required_systems = {
    "detector_only",
    "detector_plus_xai",
    "full_er_cyris",
}

if set(system_rows["system"].unique()) != required_systems:
    raise RuntimeError("Tiga sistem Paket B tidak lengkap")


# ------------------------------------------------------------
# 1. Ringkasan dengan definisi false acceptance yang konsisten
# ------------------------------------------------------------

group_keys = [
    "dataset",
    "representation",
    "model",
    "condition",
]

summary_rows = []

for keys, comparison in system_rows.groupby(
    group_keys,
    sort=True,
):
    baseline = comparison[
        comparison["system"] == "detector_only"
    ]

    baseline_n = len(baseline)

    baseline_false_count = int(
        (baseline["y_true"] == 0).sum()
    )

    baseline_false_rate = (
        baseline_false_count / max(baseline_n, 1)
    )

    for system_name, group in comparison.groupby(
        "system",
        sort=True,
    ):
        accepted = group["action"].eq("accept")
        warned = group["action"].eq("warn")
        abstained = group["action"].eq("abstain")

        positive = group["y_true"].eq(1)
        negative = group["y_true"].eq(0)

        false_accepted = negative & accepted
        true_accepted = positive & accepted

        false_accept_count = int(
            false_accepted.sum()
        )

        false_accept_rate = (
            false_accept_count / max(len(group), 1)
        )

        absolute_reduction = (
            baseline_false_rate -
            false_accept_rate
        )

        if baseline_false_count > 0:
            relative_reduction = (
                1.0 -
                false_accept_count /
                baseline_false_count
            )
        else:
            relative_reduction = np.nan

        summary_rows.append({
            **dict(zip(group_keys, keys)),
            "system": system_name,
            "n_alerts": int(len(group)),
            "accept_coverage": float(accepted.mean()),
            "warn_rate": float(warned.mean()),
            "abstain_rate": float(abstained.mean()),
            "accepted_precision": (
                float(
                    group.loc[
                        accepted,
                        "y_true"
                    ].mean()
                )
                if accepted.any()
                else np.nan
            ),
            "retained_tp": (
                float(
                    true_accepted.sum() /
                    max(positive.sum(), 1)
                )
            ),
            "false_accept_count": false_accept_count,
            "false_accept_rate_per_candidate": (
                float(false_accept_rate)
            ),
            "baseline_false_accept_count": (
                baseline_false_count
            ),
            "baseline_false_accept_rate_per_candidate": (
                float(baseline_false_rate)
            ),
            "absolute_false_accept_reduction": (
                float(absolute_reduction)
            ),
            "relative_false_accept_reduction": (
                float(relative_reduction)
                if not np.isnan(relative_reduction)
                else np.nan
            ),
            "mean_shap_seconds": float(
                group["shap_seconds"].mean()
            ),
        })


corrected_summary = pd.DataFrame(summary_rows)

temporary_summary = CORRECTED_SUMMARY.with_suffix(
    ".tmp.csv"
)

corrected_summary.to_csv(
    temporary_summary,
    index=False,
)

os.replace(
    temporary_summary,
    CORRECTED_SUMMARY,
)


# ------------------------------------------------------------
# 2. Cluster bootstrap berdasarkan run_id
# ------------------------------------------------------------

full = system_rows[
    system_rows["system"] == "full_er_cyris"
].copy()

full["negative"] = full["y_true"].eq(0)

full["false_accepted"] = (
    full["negative"] &
    full["action"].eq("accept")
)

full["positive"] = full["y_true"].eq(1)

full["true_accepted"] = (
    full["positive"] &
    full["action"].eq("accept")
)

run_statistics = (
    full.groupby("run_id", as_index=False)
    .agg(
        n=("alert_id", "size"),
        negative=("negative", "sum"),
        false_accepted=("false_accepted", "sum"),
        positive=("positive", "sum"),
        true_accepted=("true_accepted", "sum"),
    )
)

N_BOOTSTRAP = 5000
BOOTSTRAP_SEED = 20260923

rng = np.random.default_rng(
    BOOTSTRAP_SEED
)

absolute_reductions = []
relative_reductions = []
retained_true_positives = []

for _ in range(N_BOOTSTRAP):
    sampled_runs = run_statistics.iloc[
        rng.integers(
            0,
            len(run_statistics),
            len(run_statistics),
        )
    ]

    n_total = sampled_runs["n"].sum()
    negative_total = sampled_runs[
        "negative"
    ].sum()

    false_accepted_total = sampled_runs[
        "false_accepted"
    ].sum()

    positive_total = sampled_runs[
        "positive"
    ].sum()

    true_accepted_total = sampled_runs[
        "true_accepted"
    ].sum()

    absolute_reductions.append(
        (
            negative_total -
            false_accepted_total
        ) / max(n_total, 1)
    )

    relative_reductions.append(
        1.0 -
        false_accepted_total /
        max(negative_total, 1)
    )

    retained_true_positives.append(
        true_accepted_total /
        max(positive_total, 1)
    )


def bootstrap_result(values):
    values = np.asarray(
        values,
        dtype=float,
    )

    return {
        "estimate": float(values.mean()),
        "ci_2p5": float(
            np.quantile(values, 0.025)
        ),
        "ci_97p5": float(
            np.quantile(values, 0.975)
        ),
    }


bootstrap_results = {
    "method": (
        "paired cluster bootstrap by run_id"
    ),
    "bootstrap_iterations": N_BOOTSTRAP,
    "bootstrap_seed": BOOTSTRAP_SEED,
    "n_run_clusters": int(
        len(run_statistics)
    ),
    "absolute_false_accept_reduction_per_candidate": (
        bootstrap_result(
            absolute_reductions
        )
    ),
    "relative_false_accept_count_reduction": (
        bootstrap_result(
            relative_reductions
        )
    ),
    "retained_true_positive_rate": (
        bootstrap_result(
            retained_true_positives
        )
    ),
}

temporary_bootstrap = (
    CORRECTED_BOOTSTRAP.with_suffix(
        ".tmp.json"
    )
)

temporary_bootstrap.write_text(
    json.dumps(
        bootstrap_results,
        indent=2,
    ),
    encoding="utf-8",
)

os.replace(
    temporary_bootstrap,
    CORRECTED_BOOTSTRAP,
)


# ------------------------------------------------------------
# 3. Kunci analisis koreksi
# ------------------------------------------------------------

gate_lock = json.loads(
    (
        OUT_DIR /
        "GATE_EVALUATION_LOCK.json"
    ).read_text(
        encoding="utf-8"
    )
)

analysis_lock = {
    "status": "CORRECTED_ANALYSIS_COMPLETE",
    "source_evaluation_status": (
        gate_lock["status"]
    ),
    "gate_registry_sha256": (
        gate_lock["gate_registry_sha256"]
    ),
    "source_evaluation_rows": int(
        len(system_rows)
    ),
    "n_unique_alerts": int(
        system_rows["alert_id"].nunique()
    ),
    "n_run_clusters": int(
        len(run_statistics)
    ),
    "bootstrap_method": (
        "paired cluster bootstrap by run_id"
    ),
    "fixed_evaluation_used_for_tuning": False,
    "original_evaluation_modified": False,
}

temporary_lock = ANALYSIS_LOCK.with_suffix(
    ".tmp.json"
)

temporary_lock.write_text(
    json.dumps(
        analysis_lock,
        indent=2,
    ),
    encoding="utf-8",
)

os.replace(
    temporary_lock,
    ANALYSIS_LOCK,
)


print("Corrected summary:", CORRECTED_SUMMARY)
print("Corrected bootstrap:", CORRECTED_BOOTSTRAP)
print("Corrected analysis lock:", ANALYSIS_LOCK)
print(json.dumps(bootstrap_results, indent=2))

Corrected summary: /content/drive/MyDrive/ercyris_framework_validation/frozen_gate_v1/three_system_summary_corrected.csv
Corrected bootstrap: /content/drive/MyDrive/ercyris_framework_validation/frozen_gate_v1/package_b_cluster_bootstrap_corrected.json
Corrected analysis lock: /content/drive/MyDrive/ercyris_framework_validation/frozen_gate_v1/PACKAGE_B_CORRECTED_ANALYSIS_LOCK.json
{
  "method": "paired cluster bootstrap by run_id",
  "bootstrap_iterations": 5000,
  "bootstrap_seed": 20260923,
  "n_run_clusters": 360,
  "absolute_false_accept_reduction_per_candidate": {
    "estimate": 0.15418732988452102,
    "ci_2p5": 0.13187110927166756,
    "ci_97p5": 0.1780607071174336
  },
  "relative_false_accept_count_reduction": {
    "estimate": 0.6172609522114068,
    "ci_2p5": 0.5628126570279175,
    "ci_97p5": 0.6700828805624439
  },
  "retained_true_positive_rate": {
    "estimate": 0.6920439163381331,
    "ci_2p5": 0.6615213465434885,
    "ci_97p5": 0.7226012768727447
  }
}


## Interpretasi wajib

- `detector_only` dan `detector_plus_xai` harus memiliki tindakan identik; XAI tanpa gate adalah kontrol negatif untuk efek kebijakan keputusan.
- Klaim utama berasal dari perbandingan `full_er_cyris` terhadap dua kontrol pada fixed internal evaluation set.
- Agreement lintas-model hanya diagnostik untuk H4, bukan input hard gate.
- Jangan membuka ulang pencarian threshold setelah evaluasi. Jika aturan berubah, buat versi Paket B baru dan ulangi development freeze dari awal.
